En este cuaderno probamos una inferencia del modelo Gemma 3 4b antes de hacer el fine tuning.

Antes de eso debemos instalar Ollama en Colab, para lo cual instalamos la librería zstd.

In [ ]:
import subprocess
try:
    # Install zstd for Ollama installation
    install_zstd_command = ['sudo', 'apt-get', 'update', '-y']
    process = subprocess.run(install_zstd_command, check=True, capture_output=True, text=True)
    print(process.stdout)
    install_zstd_command = ['sudo', 'apt-get', 'install', 'zstd', '-y']
    process = subprocess.run(install_zstd_command, check=True, capture_output=True, text=True)
    print(process.stdout)
    print('zstd installed successfully.')
except subprocess.CalledProcessError as e:
    print(f"Error installing zstd: {e.stderr}")
    print("Please try running the command manually: `sudo apt-get install zstd`")
except FileNotFoundError:
    print("apt-get command not found. Are you on a Debian/Ubuntu-based system?")


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,959 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,588 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 12.7 MB in 1s (8,880 kB/s)
Reading packa

Ahora la podemos instalar Ollama.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q google-genai pydantic langchain-community

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Iniciamos el servidor de Ollama en segundo plano, que ha salido OK.

In [ ]:
# ==========================================
# CELDA 2: Iniciar el servidor de Ollama en segundo plano
# ==========================================
import subprocess
import time

# Lanzar el proceso de Ollama asignándole la GPU T4 de Colab
ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Esperar 5 segundos a que el demonio se inicialice
print("Servidor Ollama iniciado correctamente.")

Servidor Ollama iniciado correctamente.


Descargamos el modelo Gemma 3 4b desde Ollama.

In [ ]:
# ==========================================
# CELDA 3: Descargar los modelos requeridos (Gemma 3 4B y Llama 3.2 3B)
# ==========================================
# Generador principal (Base para evaluar zero-shot / futuro Fine-Tuning)
!ollama pull gemma3:4b

Ahora descargamos un prompt para hacer el comentario, concretamente el del 24 de agosto, a ver qué contesta el modelo...

In [ ]:
# Ajusta la ruta a la ubicación de tu archivo en Drive
ruta_prompt = "/content/drive/MyDrive/Ollama_app_comentario/prompt_comentario_mercados_24ago.txt"

with open(ruta_prompt, "r", encoding="utf-8") as f:
    prompt = f.read()

print("Prompt cargado desde Drive.")

Prompt cargado desde Drive.


Añadimos una coda al final del prompt para asegurarnos que contesta en castellano.

In [ ]:
anclaje_final = """

---
TAREA OBLIGATORIA:
Redacta AHORA el comentario financiero de 650 palabras en ESPAÑOL.
NO resumas las noticias por puntos. NO respondas en inglés.
Empieza directamente con el primer párrafo en negrita.
"""

In [ ]:
prompt += anclaje_final

El prompt del sistema va a ser fijo para todos los comentarios, y es el siguiente:

In [ ]:
prompt_sistema = ("Eres un analista financiero senior. Debes redactar un comentario de mercados en castellano (650 palabras aprox.)"
 "basado en datos reales y las noticias proporcionadas."
"## FORMATO:"
"- Estilo profesional, conciso y analítico."
"- Usa abreviaturas: EEUU, UK, ATH, yoy, pbs, BBG."
"- Porcentajes con signo: +2.3%, -1.5%."
"- El comentario debe tener 4-5 párrafos en los que trates, al menos, los siguientes temas, "
"sin que sea este un orden de importancia:"
   " - Renta Variable, principalmente norteamericana y en menor medida europea"
   " - Materias Primas"
   " - Renta Fija, tipos de interés, Reserva Federal"
   " - Noticias corporativas de primer orden en EEUU"
   " - Agenda de datos a conocerse: Resultados empresariales en EEUU y Europa de empresas especialmente importantes, "
   "así como datos macro, especialmente inflación y desempleo en EEUU, UK y la Eurozona"
"- Los temas tratados en cada párrafo irán de más a menos importancia, y su importancia dependerá de las "
"fuentes Before The European Bell y Five Things, que se te adjuntan en el resto de este prompt."
"## INSTRUCCIONES FINALES:"
"1. Usa los DATOS NUMÉRICOS EXACTOS que se te han dado. NO los inventes."
"2. Genera los párrafos temáticos en orden de importancia: primero los que consideres más relevantes."
"3. La información de Before the Bell y Five Things es la base para el análisis cualitativo,"
"las perspectivas y las noticias corporativas."
"4. Si se incluyen discursos de la FED, son la fuente principal para hablar de política monetaria."
"5. Incluye un breve resumen de las expectativas de tipos de la Fed, BCE y BOE en el apartado de"
"Renta Fija, tipos de interés, y Reserva Federal"
"6. Si hoy es lunes, los ratios PE, los crecimientos de EPS y las tendencias se tienen "
"que integrar en el análisis de renta variable."
"7. Los datos de resultados empresariales y macro a conocerse deben ir hacia el final de comentario, "
"a menos que tengan conexión con el análisis de renta variable o renta fija."
"8. Pon en negrita los nombres de países, empresas, datos macro, y personalidades importantes."
"9. No añadas texto introductorio como Claro, aquí tienes... . Empieza directamente con el análisis."
"Genera el comentario de mercados a continuación:" )

In [ ]:
import urllib.request
import json

Esta es el ejemplo de inferencia del modelo Gemma 3 4b.

In [ ]:
def consultar_ollama(model: str, prompt: str, system_prompt: str = ""):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "system": system_prompt,
        "stream": False,
        "options": {
            "temperature": 0.1, # Baja temperatura para evitar invención de datos
            "num_ctx": 4096     # Ampliar ventana de contexto para procesar las noticias
        }
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json'}
    )
    with urllib.request.urlopen(req) as response:
        res = json.loads(response.read().decode('utf-8'))
        return res['response']

# Ejecución de la inferencia zero-shot en Gemma 3 4B
resultado_gemma = consultar_ollama(
    model="gemma3:4b",
    prompt=prompt,
    system_prompt=prompt_sistema
)

print("--- RESPUESTA ZERO-SHOT (GEMMA 3 4B) ---")
print(resultado_gemma)

--- RESPUESTA ZERO-SHOT (GEMMA 3 4B) ---
**El panorama económico global se presenta hoy como un mosaico de tensiones y oportunidades, marcado por la incertidumbre política, la evolución de los mercados energéticos y las estrategias de inversión que se vislumbran en diversos sectores. La reciente sesión ha estado dominada por la persistente crisis geopolítica, la recuperación de los precios del petróleo y las reacciones del mercado a las noticias económicas y financieras que se han ido desplegando a lo largo de la jornada.**

La situación en el frente político sigue siendo el principal motor de volatilidad. La decisión de China de continuar apoyando un fin diplomático de la guerra entre Estados Unidos e Irán, aunque tardía en su publicación, refleja la preocupación global por las consecuencias de este conflicto y su impacto en las rutas marítimas clave. La amenaza de represalias económicas por parte de Washington, que se materializará en medidas para castigar a los socios comerciales de

Cerramos el servidor de Ollama.

In [ ]:
# Terminar el proceso de Ollama
ollama_process.terminate() # O ollama_process.kill() para forzarlo
ollama_process.wait()      # Liberar recursos del sistema

print("Servidor Ollama detenido correctamente.")

Servidor Ollama detenido correctamente.


Este es una rutina para generar comentarios, que finalmente no utilizamos. Los comentarios a generar lo serán cuando evaluemos el fine tuning con Ragas.

In [ ]:
import json
import os

# Directorios donde guardas tus archivos diarios
DIR_PROMPTS = "./datos/prompts"
DIR_COMENTARIOS = "./datos/comentarios"
OUTPUT_JSONL = "dataset_market_commentary.jsonl"

system_prompt_base = (
    "Eres un analista financiero senior. Debes redactar un comentario de mercados en castellano "
    "(650 palabras aprox.) basado en datos reales y las noticias proporcionadas. Usa estilo profesional, "
    "abreviaturas financieras (EEUU, UK, ATH, yoy, pbs, BBG) y pon en negrita nombres de países, "
    "empresas, datos macro y personalidades."
)

dataset = []

# Recorrer los archivos de prompts
for filename in os.listdir(DIR_PROMPTS):
    if filename.endswith(".txt"):
        prompt_path = os.path.join(DIR_PROMPTS, filename)
        comentario_path = os.path.join(DIR_COMENTARIOS, filename)

        # Verificar que existe el par prompt - comentario
        if os.path.exists(comentario_path):
            with open(prompt_path, "r", encoding="utf-8") as f_user:
                user_content = f_user.read().strip()

            with open(comentario_path, "r", encoding="utf-8") as f_assistant:
                assistant_content = f_assistant.read().strip()

            # Formatear el registro para ChatML
            registro = {
                "messages": [
                    {"role": "system", "content": system_prompt_base},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content}
                ]
            }
            dataset.append(registro)

# Guardar en archivo .jsonl
with open(OUTPUT_JSONL, "w", encoding="utf-8") as f_out:
    for entry in dataset:
        f_out.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ ¡Dataset generado con éxito! Total de ejemplos procesados: {len(dataset)}")